# 20 Newsgroups — Dataset Inspection, Preprocessing, EDA & Feature Extraction

**Author:** Reetikesh Choudhury  
**Module:** Dataset Loading, Preprocessing & Split (own component)  

This notebook loads the `twenty+newsgroups.zip` dataset, runs full exploratory data analysis,
creates a stratified **80/20 train/test split**, and performs **TF-IDF feature extraction** analysis.

**Data flow (no leakage):**
1. Load all documents from `twenty+newsgroups.zip` → `20_newsgroups.tar.gz`
2. Basic text cleaning / preprocessing (token-level — no corpus statistics)
3. Create stratified 80/20 split (`train_test_split`, `random_state=42`)
4. TF-IDF vectorizer fitted **only** on training documents

**Sections:**
1. Dataset Loading & Shape  
2. Full Dataset Analysis Report  
3. Category Distribution  
4. Stratified 80/20 Train/Test Split  
5. Document Length Distribution (character-level)  
6. Preprocessing Pipeline & Token-Level EDA  
7. Vocabulary Analysis  
8. Zipf's Law — Rank-Frequency Distribution  
9. Top Terms per Category  
10. TF-IDF Feature Extraction Analysis  
11. Save Results

## 0. Setup — imports and paths

In [ ]:
import sys
from pathlib import Path

# Make src/ importable from the notebooks/ directory
SRC_DIR = Path().resolve().parent / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import collections
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from data_loader  import load_dataset, load_dataframe, dataset_info, CATEGORIES
from data_analysis import analyse_dataset, save_results, print_report
from preprocessing import preprocess, preprocess_documents, pipeline_steps

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

ARCHIVE_PATH = Path().resolve().parent / 'twenty+newsgroups.zip'
RESULTS_DIR  = Path().resolve().parent / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

print('Archive exists:', ARCHIVE_PATH.exists())
print('Archive path  :', ARCHIVE_PATH)
print('Results dir   :', RESULTS_DIR)

## 1. Dataset Loading — Quick Shape Check

The archive contains `20_newsgroups.tar.gz` with a directory-per-document layout:
```
20_newsgroups/<category>/<article_id>
```
There is **no pre-assigned train/test split** — the 80/20 split is created in Section 4.

In [ ]:
# Fast metadata scan (no full text parse)
info = dataset_info(ARCHIVE_PATH)
print(f"Categories in archive : {info['num_categories']}")
print(f"Total raw documents   : {info['total_documents']:,}")
print()
print('Documents per category (raw file count):')
for cat, n in info['counts'].items():
    print(f'  {cat:<35} {n:,}')

In [ ]:
# Full load — all documents, headers stripped
docs = load_dataset(ARCHIVE_PATH)
print(f"Loaded {len(docs):,} documents")
print(f"Keys   : {list(docs[0].keys())}")
print()
print('Sample record:')
sample = docs[500]
for k, v in sample.items():
    val = repr(v[:80] + '…') if isinstance(v, str) and len(v) > 80 else v
    print(f'  {k:<10}: {val}')

In [ ]:
# Load as DataFrame for easy exploration
df_raw = load_dataframe(ARCHIVE_PATH)
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

## 2. Full Dataset Analysis Report

In [ ]:
result = analyse_dataset(ARCHIVE_PATH)
print_report(result)

## 3. Category Distribution

In [ ]:
cat_counts = df_raw['category'].value_counts().sort_index()
cat_df = cat_counts.reset_index()
cat_df.columns = ['category', 'count']
print(f"Total documents: {cat_df['count'].sum():,}")
print(f"Min per category: {cat_df['count'].min()}  Max: {cat_df['count'].max()}")
cat_df

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(cat_df['category'], cat_df['count'], color=sns.color_palette('muted', 20))
ax.set_xticklabels(cat_df['category'], rotation=45, ha='right', fontsize=8)
ax.set_title('Documents per Category (full dataset — 20 newsgroups)', fontsize=13)
ax.set_ylabel('Document count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=6)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'docs_per_category.png', dpi=120)
plt.show()
print('Class balance — std dev:', round(cat_df["count"].std(), 1),
      '  CV:', round(cat_df["count"].std() / cat_df["count"].mean(), 3))

## 4. Stratified 80/20 Train/Test Split

Split parameters: `test_size=0.20`, `random_state=42`, `stratify=labels`  
**Data leakage note:** TF-IDF is fitted only on training documents (Section 10).

In [ ]:
labels = df_raw['label'].tolist()

train_df, test_df = train_test_split(
    df_raw,
    test_size=0.20,
    random_state=42,
    stratify=labels,
)

total   = len(df_raw)
train_n = len(train_df)
test_n  = len(test_df)

print('=' * 52)
print('  80/20 STRATIFIED SPLIT RESULTS')
print('=' * 52)
print(f'  Total documents   : {total:,}')
print(f'  Training documents: {train_n:,}  ({100 * train_n / total:.1f}%)')
print(f'  Testing documents : {test_n:,}  ({100 * test_n / total:.1f}%)')
print(f'  Split params      : test_size=0.20, random_state=42, stratify=labels')
print('=' * 52)

In [ ]:
# Per-category distribution table
train_cats = train_df['category'].value_counts().sort_index()
test_cats  = test_df['category'].value_counts().sort_index()

split_dist = pd.DataFrame({'train': train_cats, 'test': test_cats})
split_dist['total']     = split_dist['train'] + split_dist['test']
split_dist['train_pct'] = (split_dist['train'] / split_dist['total'] * 100).round(1)
split_dist['test_pct']  = (split_dist['test']  / split_dist['total'] * 100).round(1)

print(f"{'Category':<35} {'Train':>6}  {'Test':>5}  {'Total':>6}  {'Train%':>7}  {'Test%':>6}")
print('-' * 72)
for cat, row in split_dist.iterrows():
    print(f"{cat:<35} {int(row['train']):>6}  {int(row['test']):>5}  "
          f"{int(row['total']):>6}  {row['train_pct']:>6.1f}%  {row['test_pct']:>5.1f}%")
print('-' * 72)
print(f"{'TOTAL':<35} {int(split_dist['train'].sum()):>6}  "
      f"{int(split_dist['test'].sum()):>5}  {int(split_dist['total'].sum()):>6}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
x = range(len(split_dist))
width = 0.4
ax.bar([i - width/2 for i in x], split_dist['train'], width, label='Train (~80%)')
ax.bar([i + width/2 for i in x], split_dist['test'],  width, label='Test (~20%)')
ax.set_xticks(list(x))
ax.set_xticklabels(split_dist.index, rotation=45, ha='right', fontsize=8)
ax.set_title('Train / Test document counts per category (80/20 stratified split)')
ax.set_ylabel('Documents')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'split_balance.png', dpi=120)
plt.show()

## 4b. Split Ratio Comparison — 80/20 vs 75/25 vs 70/30

Compare three common train/test split ratios on the same dataset using the same `random_state=42` and `stratify=labels`.
All three use sklearn `train_test_split`.

**What to look for:**
- How many documents each ratio allocates to training vs testing
- Whether stratification holds across all three (train% ≈ target for every category)
- The trade-off: more training data → better model; more test data → more reliable evaluation

In [1]:
# ── Split ratio comparison ──────────────────────────────────────────────────
splits_to_compare = [
    ('80/20', 0.20),
    ('75/25', 0.25),
    ('70/30', 0.30),
]

split_results = {}
for label_str, test_size in splits_to_compare:
    tr, te = train_test_split(
        df_raw,
        test_size=test_size,
        random_state=42,
        stratify=df_raw['label'].tolist(),
    )
    split_results[label_str] = {'train': tr, 'test': te}

# ── Overall counts table ────────────────────────────────────────────────────
print('=' * 60)
print('  SPLIT RATIO COMPARISON (random_state=42, stratify=label)')
print('=' * 60)
print(f"{'Ratio':<8} {'Train':>7} {'Train%':>8}  {'Test':>6} {'Test%':>7}  {'Total':>7}")
print('-' * 60)
for label_str, d in split_results.items():
    n_tr = len(d['train'])
    n_te = len(d['test'])
    n_tot = n_tr + n_te
    print(f"{label_str:<8} {n_tr:>7,} {100*n_tr/n_tot:>7.1f}%  "
          f"{n_te:>6,} {100*n_te/n_tot:>6.1f}%  {n_tot:>7,}")
print('=' * 60)

  SPLIT RATIO COMPARISON  (random_state=42, stratify=label)
Ratio      Train   Train%    Test   Test%    Total
--------------------------------------------------------------
80/20     15,968    80.0%   3,993   20.0%   19,961
75/25     14,970    75.0%   4,991   25.0%   19,961
70/30     13,972    70.0%   5,989   30.0%   19,961


In [2]:
# ── Per-category stratification table for all three ratios ─────────────────
import pandas as pd

rows = []
for label_str, d in split_results.items():
    tr_cats = d['train']['category'].value_counts().sort_index()
    te_cats = d['test']['category'].value_counts().sort_index()
    for cat in CATEGORIES:
        tr_n = int(tr_cats.get(cat, 0))
        te_n = int(te_cats.get(cat, 0))
        tot  = tr_n + te_n
        rows.append({
            'split_ratio' : label_str,
            'category'    : cat,
            'train'       : tr_n,
            'test'        : te_n,
            'total'       : tot,
            'train_pct'   : round(100 * tr_n / tot, 1) if tot else 0,
        })

comp_df = pd.DataFrame(rows)

# Pivot: one row per category, columns grouped by ratio
pivot = comp_df.pivot_table(
    index='category',
    columns='split_ratio',
    values=['train', 'test', 'train_pct'],
    aggfunc='first',
)
# Flatten multi-level columns
pivot.columns = [f'{val}_{ratio}' for val, ratio in pivot.columns]
pivot = pivot.reindex(CATEGORIES)  # canonical category order

# Pretty print
print(f"{'Category':<35}",
      f"{'── 80/20 ──':^18}",
      f"{'── 75/25 ──':^18}",
      f"{'── 70/30 ──':^18}")
print(f"{'':35} {'Train':>6} {'Test':>5} {'Tr%':>5}",
      f"  {'Train':>6} {'Test':>5} {'Tr%':>5}",
      f"  {'Train':>6} {'Test':>5} {'Tr%':>5}")
print('-' * 95)
for cat in CATEGORIES:
    r = pivot.loc[cat]
    print(f"{cat:<35}",
          f"{int(r['train_80/20']):>6}  {int(r['test_80/20']):>5} {r['train_pct_80/20']:>5.1f}%",
          f"  {int(r['train_75/25']):>6}  {int(r['test_75/25']):>5} {r['train_pct_75/25']:>5.1f}%",
          f"  {int(r['train_70/30']):>6}  {int(r['test_70/30']):>5} {r['train_pct_70/30']:>5.1f}%")
print('-' * 95)
totals_row = comp_df.groupby('split_ratio')[['train','test']].sum()
for lbl in ['80/20','75/25','70/30']:
    tr = totals_row.loc[lbl,'train']; te = totals_row.loc[lbl,'test']
    print(f"{'TOTAL  (' + lbl + ')':>54}  {tr:>6}  {te:>5}")

Category                                   ── 80/20 ──           ── 75/25 ──           ── 70/30 ──     
                                      Train  Test    Tr%   Train  Test    Tr%   Train  Test    Tr%
---------------------------------------------------------------------------------------------------
alt.atheism                             799   200  80.0%     749   250  75.0%     699   300  70.0%
comp.graphics                           798   200  80.0%     748   250  74.9%     699   299  70.0%
comp.os.ms-windows.misc                 794   198  80.0%     744   248  75.0%     694   298  70.0%
comp.sys.ibm.pc.hardware                798   200  80.0%     749   249  75.1%     699   299  70.0%
comp.sys.mac.hardware                   797   199  80.0%     747   249  75.0%     697   299  70.0%
comp.windows.x                          798   200  80.0%     749   249  75.1%     699   299  70.0%
misc.forsale                            795   198  80.1%     745   248  75.0%     695   298  70.0%
rec.

In [3]:
# ── SINGLE-CELL COMPARISON SUMMARY — all three splits at a glance ──────────
print('=' * 60)
print('  COMPLETE SPLIT COMPARISON SUMMARY')
print('=' * 60)
print(f"  {'Metric':<28} {'80/20':>9}  {'75/25':>9}  {'70/30':>9}")
print('  ' + '-' * 57)

for label_str, d in split_results.items():
    n_tr = len(d['train']); n_te = len(d['test'])

counts_row  = {lbl: (len(d['train']), len(d['test'])) for lbl, d in split_results.items()}
devs_row    = {}
for lbl, test_size in splits_to_compare:
    d = split_results[lbl]
    tr_c = d['train']['category'].value_counts()
    te_c = d['test']['category'].value_counts()
    target = 100 * (1 - test_size)
    devs = []
    for cat in CATEGORIES:
        tr = tr_c.get(cat,0); te = te_c.get(cat,0); tot = tr+te
        if tot: devs.append(abs(100*tr/tot - target))
    devs_row[lbl] = max(devs)

tr_vals = [counts_row[l][0] for l in ['80/20','75/25','70/30']]
te_vals = [counts_row[l][1] for l in ['80/20','75/25','70/30']]
tot     = tr_vals[0] + te_vals[0]

print(f"  {'Training documents':<28} {tr_vals[0]:>9,}  {tr_vals[1]:>9,}  {tr_vals[2]:>9,}")
print(f"  {'Testing  documents':<28} {te_vals[0]:>9,}  {te_vals[1]:>9,}  {te_vals[2]:>9,}")
print(f"  {'Train %':<28} {100*tr_vals[0]/tot:>8.1f}%  {100*tr_vals[1]/tot:>8.1f}%  {100*tr_vals[2]/tot:>8.1f}%")
print(f"  {'Test  %':<28} {100*te_vals[0]/tot:>8.1f}%  {100*te_vals[1]/tot:>8.1f}%  {100*te_vals[2]/tot:>8.1f}%")
print(f"  {'Max category deviation':<28} {devs_row['80/20']:>8.2f}%  {devs_row['75/25']:>8.2f}%  {devs_row['70/30']:>8.2f}%")
print(f"  {'Stratification':<28} {'OK':>9}  {'OK':>9}  {'OK':>9}")
print('=' * 60)
print('  Project uses: 80/20  (most training data, best model quality)')
print('=' * 60)

  COMPLETE SPLIT COMPARISON SUMMARY
  Metric                        80/20       75/25       70/30
  ---------------------------------------------------------
  Training documents           15,968      14,970      13,972
  Testing  documents            3,993       4,991       5,989
  Train %                       80.0%       75.0%       70.0%
  Test  %                       20.0%       25.0%       30.0%
  Max category deviation         0.06%       0.05%       0.04%
  Stratification                  OK          OK          OK
  Project uses: 80/20  (most training data, best model quality)


In [ ]:
# ── Visualisation ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
colors = {'train': 'steelblue', 'test': 'tomato'}

for ax, (label_str, d) in zip(axes, split_results.items()):
    tr_cats = d['train']['category'].value_counts().sort_index()
    te_cats = d['test']['category'].value_counts().sort_index()
    x = np.arange(len(CATEGORIES))
    w = 0.4
    ax.bar(x - w/2, [tr_cats.get(c, 0) for c in CATEGORIES], w,
           label='Train', color=colors['train'], alpha=0.85)
    ax.bar(x + w/2, [te_cats.get(c, 0) for c in CATEGORIES], w,
           label='Test',  color=colors['test'],  alpha=0.85)
    ax.set_title(f'Split {label_str}', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(CATEGORIES, rotation=55, ha='right', fontsize=6.5)
    ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
    # Annotate total train / test
    n_tr = len(d['train']); n_te = len(d['test'])
    ax.set_xlabel(f'Train={n_tr:,}  Test={n_te:,}  ({100*n_tr/(n_tr+n_te):.0f}%/{100*n_te/(n_tr+n_te):.0f}%)',
                  fontsize=9)

axes[0].set_ylabel('Documents')
fig.suptitle(
    'Per-category document counts for three split ratios (random_state=42, stratify=label)',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'split_ratio_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary bar chart: train vs test absolute counts per ratio ──────────────
ratios  = list(split_results.keys())
tr_totals = [len(split_results[r]['train']) for r in ratios]
te_totals = [len(split_results[r]['test'])  for r in ratios]

x = np.arange(len(ratios))
w = 0.35
fig, ax = plt.subplots(figsize=(7, 5))
b1 = ax.bar(x - w/2, tr_totals, w, label='Train', color='steelblue', alpha=0.88)
b2 = ax.bar(x + w/2, te_totals, w, label='Test',  color='tomato',    alpha=0.88)

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(ratios, fontsize=12)
ax.set_title('Train vs Test document count by split ratio\n(total = 19,961 documents)', fontsize=12)
ax.set_ylabel('Documents')
ax.set_xlabel('Split ratio (Train/Test)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.legend(fontsize=11)
ax.set_ylim(0, max(tr_totals) * 1.12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'split_ratio_summary.png', dpi=120)
plt.show()

print('\nConclusion:')
print(f'  80/20 gives the most training data ({tr_totals[0]:,} docs) — chosen for this project.')
print(f'  70/30 gives the most test data ({te_totals[2]:,} docs) — more reliable evaluation.')
print(f'  75/25 is the middle ground ({tr_totals[1]:,} train / {te_totals[1]:,} test).')
print(f'  All three achieve near-perfect stratification (train% ≈ target ± 0.1% per category).')

## 5. Document Length Distribution (character-level)

In [ ]:
df_raw['text_len'] = df_raw['text'].str.len()
print('Document length statistics (characters):')
print(df_raw['text_len'].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_raw['text_len'].clip(upper=10000), bins=60, edgecolor='none', color='steelblue')
axes[0].set_title('Character length distribution (clipped at 10k)')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Documents')

axes[1].hist(np.log1p(df_raw['text_len']), bins=60, edgecolor='none', color='steelblue')
axes[1].set_title('Character length distribution (log scale)')
axes[1].set_xlabel('log(1 + characters)')
axes[1].set_ylabel('Documents')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'length_distribution.png', dpi=120)
plt.show()

In [ ]:
avg_len = df_raw.groupby('category')['text_len'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(avg_len.index, avg_len.values, color=sns.color_palette('muted', 20))
ax.set_xticklabels(avg_len.index, rotation=45, ha='right', fontsize=8)
ax.set_title('Average document length per category (characters)')
ax.set_ylabel('Avg characters')
ax.axhline(df_raw['text_len'].mean(), color='red', linestyle='--', linewidth=1, label='Overall mean')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'avg_length_per_category.png', dpi=120)
plt.show()

## 6. Preprocessing Pipeline & Token-Level EDA

Apply the 6-step preprocessing pipeline to all documents and compare:
- Raw token count vs. clean token count (preprocessing impact)
- Token length distribution after preprocessing
- Stopword removal and stemming effectiveness

In [ ]:
# Demo: step-by-step pipeline on one document
sample_text = docs[0]['text']
steps = pipeline_steps(sample_text)

print('=== PREPROCESSING PIPELINE — STEP BY STEP ===')
print(f"\nStep 0 – Raw text ({len(sample_text)} chars):")
print(sample_text[:300].replace('\n', ' ') + '...')
print(f"\nStep 1 – After header removal ({len(steps['no_headers'])} chars):")
print(steps['no_headers'][:300].replace('\n', ' ') + '...')
print(f"\nStep 2 – Lowercased (first 200 chars): {steps['lowercased'][:200]}...")
print(f"\nStep 3 – Tokens ({len(steps['tokens'])} total): {steps['tokens'][:20]} ...")
print(f"\nStep 4 – After stopword removal ({len(steps['no_stopwords'])} tokens): {steps['no_stopwords'][:20]} ...")
print(f"\nStep 5 – After stemming ({len(steps['stemmed'])} tokens): {steps['stemmed'][:20]} ...")
print(f"\nStep 6 – Final clean_text ({len(steps['clean_text'].split())} tokens):")
print(steps['clean_text'][:200] + '...')

In [ ]:
# Preprocessing impact: raw tokens vs clean tokens across all documents
# (sample 2000 docs for speed if needed — here we do all)
import time
print('Preprocessing all documents for token-level analysis...')
t0 = time.perf_counter()

processed_docs = preprocess_documents(docs, verbose=True)

elapsed = time.perf_counter() - t0
print(f'Done in {elapsed:.1f}s')

df_proc = pd.DataFrame({
    'doc_id'    : [d['doc_id']   for d in processed_docs],
    'category'  : [d['category'] for d in processed_docs],
    'label'     : [d['label']    for d in processed_docs],
    'raw_tokens': [len(d['text'].split()) for d in processed_docs],
    'clean_tokens': [len(d['clean_text'].split()) if d.get('clean_text') else 0
                     for d in processed_docs],
})
df_proc['tokens_removed'] = df_proc['raw_tokens'] - df_proc['clean_tokens']
df_proc['retention_pct']  = (df_proc['clean_tokens'] / df_proc['raw_tokens'].replace(0,1) * 100).round(1)

print(f"\nDocuments with clean_text == '' (all tokens removed): "
      f"{(df_proc['clean_tokens'] == 0).sum()}")
print("\nToken count statistics:")
print(df_proc[['raw_tokens', 'clean_tokens', 'tokens_removed', 'retention_pct']].describe().round(1))

In [ ]:
# Token-length distribution (post-preprocessing) — much more IR-relevant than char count
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_proc['clean_tokens'].clip(upper=300), bins=60, edgecolor='none', color='darkorange')
axes[0].set_title('Token count per document after preprocessing (clipped at 300)')
axes[0].set_xlabel('Tokens')
axes[0].set_ylabel('Documents')
axes[0].axvline(df_proc['clean_tokens'].median(), color='red', linestyle='--',
                linewidth=1.2, label=f'Median={df_proc["clean_tokens"].median():.0f}')
axes[0].legend()

axes[1].hist(df_proc['retention_pct'], bins=40, edgecolor='none', color='darkorange')
axes[1].set_title('Token retention % after preprocessing')
axes[1].set_xlabel('% tokens kept')
axes[1].set_ylabel('Documents')
axes[1].axvline(df_proc['retention_pct'].mean(), color='red', linestyle='--',
                linewidth=1.2, label=f'Mean={df_proc["retention_pct"].mean():.1f}%')
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'token_length_distribution.png', dpi=120)
plt.show()

print(f"Average raw tokens   : {df_proc['raw_tokens'].mean():.1f}")
print(f"Average clean tokens : {df_proc['clean_tokens'].mean():.1f}")
print(f"Average retention    : {df_proc['retention_pct'].mean():.1f}%")

In [ ]:
# Average clean token count per category
avg_tokens_cat = df_proc.groupby('category')['clean_tokens'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(avg_tokens_cat.index, avg_tokens_cat.values, color=sns.color_palette('muted', 20))
ax.set_xticklabels(avg_tokens_cat.index, rotation=45, ha='right', fontsize=8)
ax.set_title('Average clean token count per category (after preprocessing)')
ax.set_ylabel('Avg tokens')
ax.axhline(df_proc['clean_tokens'].mean(), color='red', linestyle='--',
           linewidth=1, label=f'Overall mean = {df_proc["clean_tokens"].mean():.0f}')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'avg_tokens_per_category.png', dpi=120)
plt.show()

## 7. Vocabulary Analysis

Build the corpus vocabulary from the **training set only** (no leakage).  
Examine vocabulary size, hapax legomena (rare terms), and most/least frequent tokens.

In [ ]:
# Build train/test split on processed docs
proc_labels = [d['label'] for d in processed_docs]
train_proc, test_proc = train_test_split(
    processed_docs, test_size=0.20, random_state=42, stratify=proc_labels
)

print(f'Train processed docs: {len(train_proc):,}')
print(f'Test  processed docs: {len(test_proc):,}')

# Build vocabulary from training corpus only
train_token_counts = collections.Counter()
for doc in train_proc:
    tokens = doc.get('clean_text', '').split()
    train_token_counts.update(tokens)

vocab_size    = len(train_token_counts)
total_tokens  = sum(train_token_counts.values())
hapax_count   = sum(1 for v in train_token_counts.values() if v == 1)
rare_count    = sum(1 for v in train_token_counts.values() if v <= 5)

print()
print('=' * 50)
print('  TRAINING CORPUS VOCABULARY STATISTICS')
print('=' * 50)
print(f'  Unique stemmed terms   : {vocab_size:,}')
print(f'  Total token occurrences: {total_tokens:,}')
print(f'  Hapax legomena (freq=1): {hapax_count:,}  ({100*hapax_count/vocab_size:.1f}% of vocab)')
print(f'  Rare terms (freq<=5)   : {rare_count:,}  ({100*rare_count/vocab_size:.1f}% of vocab)')
print(f'  Avg occ per unique term: {total_tokens/vocab_size:.1f}')
print('=' * 50)

In [ ]:
# Top-50 most frequent terms in the training corpus
top50 = train_token_counts.most_common(50)
top50_df = pd.DataFrame(top50, columns=['term', 'count'])

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(top50_df['term'], top50_df['count'], color='steelblue')
ax.set_xticklabels(top50_df['term'], rotation=60, ha='right', fontsize=7)
ax.set_title('Top 50 most frequent terms in training corpus (after preprocessing)')
ax.set_ylabel('Frequency')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'top50_terms.png', dpi=120)
plt.show()
print(top50_df.to_string(index=False))

## 8. Zipf's Law — Rank-Frequency Distribution

Zipf's Law predicts that in natural language corpora, the frequency of a term is inversely
proportional to its rank: `freq ∝ 1/rank`.  On a log-log scale this appears as a straight line.
Deviation from this line reveals the effect of preprocessing (stopword removal + stemming).

In [ ]:
# Rank-frequency plot (Zipf's law)
sorted_freqs = sorted(train_token_counts.values(), reverse=True)
ranks = np.arange(1, len(sorted_freqs) + 1)

# Ideal Zipf line: freq = C / rank, where C = max_freq
C = sorted_freqs[0]
zipf_ideal = C / ranks

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Linear scale (clipped to top 500)
axes[0].plot(ranks[:500], sorted_freqs[:500], color='steelblue', linewidth=1.5, label='Observed')
axes[0].plot(ranks[:500], zipf_ideal[:500], color='red', linestyle='--', linewidth=1.2, label="Zipf's ideal")
axes[0].set_title("Rank-Frequency (top 500 terms, linear scale)")
axes[0].set_xlabel('Rank')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Log-log scale (full vocabulary)
axes[1].loglog(ranks, sorted_freqs, color='steelblue', linewidth=1.2, label='Observed', alpha=0.8)
axes[1].loglog(ranks, zipf_ideal, color='red', linestyle='--', linewidth=1.2, label="Zipf's ideal (slope=-1)")
axes[1].set_title("Rank-Frequency (full vocab, log-log scale — Zipf's Law)")
axes[1].set_xlabel('Rank (log)')
axes[1].set_ylabel('Frequency (log)')
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'zipf_rank_frequency.png', dpi=120)
plt.show()

# Empirical slope of the log-log curve
log_ranks = np.log(ranks)
log_freqs = np.log(sorted_freqs)
slope = np.polyfit(log_ranks, log_freqs, 1)[0]
print(f"Empirical Zipf slope (log-log regression): {slope:.3f}  (ideal = -1.0)")
print(f"Vocabulary size: {vocab_size:,} unique terms after stopword removal + stemming")

## 9. Top Terms per Category

Most frequent stemmed tokens per newsgroup — reveals each category's distinctive vocabulary.  
These are raw frequency counts (not TF-IDF weighted; see Section 10 for TF-IDF top terms).

In [ ]:
# Build per-category token counts from training data only
cat_token_counts = {cat: collections.Counter() for cat in CATEGORIES}
for doc in train_proc:
    tokens = doc.get('clean_text', '').split()
    cat_token_counts[doc['category']].update(tokens)

# Print top-15 terms for each category
print('Top 15 most frequent terms per category (training corpus, after preprocessing):\n')
for cat in CATEGORIES:
    top15 = [t for t, _ in cat_token_counts[cat].most_common(15)]
    print(f'{cat:<35} : {", ".join(top15)}')

In [ ]:
# Heatmap: top-20 global terms × categories (frequency normalised per category)
top20_global = [t for t, _ in train_token_counts.most_common(20)]

# Build normalised frequency matrix
heat_data = []
for cat in CATEGORIES:
    total_cat = sum(cat_token_counts[cat].values()) or 1
    row = [cat_token_counts[cat].get(t, 0) / total_cat * 1000 for t in top20_global]  # per-mille
    heat_data.append(row)

heat_df = pd.DataFrame(heat_data, index=CATEGORIES, columns=top20_global)

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(heat_df, ax=ax, cmap='YlOrRd', fmt='.1f', annot=True, annot_kws={'size': 6},
            linewidths=0.3, cbar_kws={'label': 'per-mille frequency'})
ax.set_title('Top-20 global terms frequency per category (per-mille, training corpus)', fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'top_terms_heatmap.png', dpi=120)
plt.show()

In [ ]:
# Grid of top-10 bar charts, one per category (4 rows × 5 cols)
fig, axes = plt.subplots(4, 5, figsize=(22, 14))
axes = axes.flatten()

for i, cat in enumerate(CATEGORIES):
    top10 = cat_token_counts[cat].most_common(10)
    terms, counts = zip(*top10) if top10 else ([], [])
    axes[i].barh(list(terms)[::-1], list(counts)[::-1], color='steelblue')
    axes[i].set_title(cat, fontsize=8)
    axes[i].tick_params(axis='y', labelsize=7)
    axes[i].tick_params(axis='x', labelsize=6)

plt.suptitle('Top 10 terms per category (training corpus, after preprocessing)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'top10_terms_per_category.png', dpi=100, bbox_inches='tight')
plt.show()

## 10. TF-IDF Feature Extraction Analysis

Fit `TfidfVectorizer` on **training documents only**, then analyse:
- Document-term matrix shape and sparsity
- IDF weight distribution (which terms are most discriminating)
- Top TF-IDF terms per category (category-discriminative terms)
- TF-IDF weight distribution across all non-zero entries

In [ ]:
# Build TF-IDF index on training documents only (no leakage)
train_texts = [d.get('clean_text', '') for d in train_proc]
test_texts  = [d.get('clean_text', '') for d in test_proc]

vectorizer = TfidfVectorizer(
    token_pattern=r'\S+',
    lowercase=False,
    sublinear_tf=True,   # 1 + log(tf)
    min_df=1,
    max_df=1.0,
    norm='l2',
)

# fit_transform on TRAIN only
train_matrix = vectorizer.fit_transform(train_texts)
# transform TEST with the already-fitted vectorizer
test_matrix  = vectorizer.transform(test_texts)

vocab      = vectorizer.vocabulary_          # term → column index
terms      = vectorizer.get_feature_names_out()
idf_values = vectorizer.idf_

print('=' * 56)
print('  TF-IDF FEATURE EXTRACTION SUMMARY')
print('=' * 56)
print(f'  Training matrix shape  : {train_matrix.shape[0]:,} docs × {train_matrix.shape[1]:,} terms')
print(f'  Test matrix shape      : {test_matrix.shape[0]:,} docs × {test_matrix.shape[1]:,} terms')
n_nonzero = train_matrix.nnz
n_total   = train_matrix.shape[0] * train_matrix.shape[1]
print(f'  Non-zero train entries : {n_nonzero:,}')
print(f'  Sparsity               : {100*(1 - n_nonzero/n_total):.2f}% sparse')
print(f'  Vocabulary size        : {len(vocab):,} unique stemmed terms')
print(f'  IDF range              : [{idf_values.min():.3f}, {idf_values.max():.3f}]')
print(f'  Avg non-zero per doc   : {n_nonzero / train_matrix.shape[0]:.1f} terms/doc')
print('=' * 56)

In [ ]:
# IDF weight distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(idf_values, bins=60, color='purple', edgecolor='none', alpha=0.8)
axes[0].set_title('IDF weight distribution (all vocabulary terms)')
axes[0].set_xlabel('IDF weight')
axes[0].set_ylabel('Terms')
axes[0].axvline(idf_values.mean(), color='red', linestyle='--',
                linewidth=1.2, label=f'Mean IDF = {idf_values.mean():.2f}')
axes[0].legend()

# Top-30 highest IDF terms (most discriminating — appear in fewest docs)
top_idf_idx  = np.argsort(idf_values)[-30:][::-1]
top_idf_terms  = terms[top_idf_idx]
top_idf_vals   = idf_values[top_idf_idx]
axes[1].barh(top_idf_terms[::-1], top_idf_vals[::-1], color='purple', alpha=0.8)
axes[1].set_title('Top 30 highest-IDF terms (most discriminating)')
axes[1].set_xlabel('IDF weight')
axes[1].tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'idf_distribution.png', dpi=120)
plt.show()

print('\nTop 30 highest-IDF terms:')
for term, idf in zip(top_idf_terms, top_idf_vals):
    print(f'  {term:<20} IDF={idf:.4f}')

In [ ]:
# Lowest IDF terms (most common across all docs — near-stopwords after preprocessing)
bot_idf_idx   = np.argsort(idf_values)[:20]
bot_idf_terms = terms[bot_idf_idx]
bot_idf_vals  = idf_values[bot_idf_idx]
print('Bottom 20 lowest-IDF terms (appear in nearly every document):')
for term, idf in zip(bot_idf_terms, bot_idf_vals):
    print(f'  {term:<20} IDF={idf:.4f}')

In [ ]:
# TF-IDF weight distribution across all non-zero entries in the training matrix
nonzero_weights = train_matrix.data  # non-zero values only

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(nonzero_weights, bins=80, color='teal', edgecolor='none', alpha=0.85)
axes[0].set_title(f'TF-IDF weight distribution (non-zero entries, n={len(nonzero_weights):,})')
axes[0].set_xlabel('TF-IDF weight')
axes[0].set_ylabel('Count')
axes[0].axvline(nonzero_weights.mean(), color='red', linestyle='--',
                linewidth=1.2, label=f'Mean={nonzero_weights.mean():.3f}')
axes[0].legend()

axes[1].hist(np.log1p(nonzero_weights), bins=60, color='teal', edgecolor='none', alpha=0.85)
axes[1].set_title('TF-IDF weight distribution (log scale)')
axes[1].set_xlabel('log(1 + weight)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'tfidf_weight_distribution.png', dpi=120)
plt.show()

print(f'Non-zero weight stats:')
print(f'  Min    : {nonzero_weights.min():.6f}')
print(f'  Max    : {nonzero_weights.max():.6f}')
print(f'  Mean   : {nonzero_weights.mean():.6f}')
print(f'  Median : {np.median(nonzero_weights):.6f}')

In [ ]:
# Top TF-IDF terms per category (category-discriminative terms)
# Average TF-IDF weight per term within each category's training documents
import scipy.sparse as sp

train_labels = [d['label'] for d in train_proc]
train_cats_list = [d['category'] for d in train_proc]

print('Top 10 TF-IDF discriminative terms per category (mean TF-IDF weight, training):\n')
top_tfidf_per_cat = {}
for cat in CATEGORIES:
    cat_mask = np.array([c == cat for c in train_cats_list])
    cat_matrix = train_matrix[cat_mask]
    mean_weights = np.asarray(cat_matrix.mean(axis=0)).ravel()
    top_idx = np.argsort(mean_weights)[-10:][::-1]
    top_terms_cat = [(terms[i], round(float(mean_weights[i]), 5)) for i in top_idx]
    top_tfidf_per_cat[cat] = top_terms_cat
    term_str = ', '.join(t for t, _ in top_terms_cat)
    print(f'{cat:<35} : {term_str}')

In [ ]:
# Grid of top-10 TF-IDF terms per category
fig, axes = plt.subplots(4, 5, figsize=(22, 14))
axes = axes.flatten()

for i, cat in enumerate(CATEGORIES):
    top_items = top_tfidf_per_cat[cat]
    t_terms  = [x[0] for x in top_items]
    t_scores = [x[1] for x in top_items]
    axes[i].barh(t_terms[::-1], t_scores[::-1], color='teal', alpha=0.85)
    axes[i].set_title(cat, fontsize=8)
    axes[i].tick_params(axis='y', labelsize=7)
    axes[i].tick_params(axis='x', labelsize=6)

plt.suptitle('Top 10 TF-IDF terms per category (mean weight, training corpus)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'tfidf_top_terms_per_category.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# TF-IDF feature matrix sparsity visualisation (sample 200 docs × 500 terms)
sample_matrix = train_matrix[:200, :500]
fig, ax = plt.subplots(figsize=(12, 5))
ax.spy(sample_matrix, markersize=0.3, color='teal', aspect='auto')
ax.set_title('TF-IDF matrix sparsity (first 200 docs × first 500 terms)')
ax.set_xlabel('Term index')
ax.set_ylabel('Document index')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'tfidf_sparsity.png', dpi=120)
plt.show()

## 11. Save Results

In [ ]:
# Compile and save all stats
stats = {
    'total_documents'     : int(total),
    'train_documents'     : int(train_n),
    'test_documents'      : int(test_n),
    'train_pct'           : round(100 * train_n / total, 2),
    'test_pct'            : round(100 * test_n  / total, 2),
    'split_method'        : 'stratified train_test_split',
    'split_params'        : {'test_size': 0.20, 'random_state': 42, 'stratify': 'label'},
    'num_categories'      : int(df_raw['category'].nunique()),
    'avg_doc_len_chars'   : round(float(df_raw['text_len'].mean()), 1),
    'avg_raw_tokens'      : round(float(df_proc['raw_tokens'].mean()), 1),
    'avg_clean_tokens'    : round(float(df_proc['clean_tokens'].mean()), 1),
    'avg_token_retention' : round(float(df_proc['retention_pct'].mean()), 1),
    'vocab_size_train'    : int(vocab_size),
    'hapax_count'         : int(hapax_count),
    'hapax_pct'           : round(100 * hapax_count / vocab_size, 1),
    'tfidf_matrix_shape'  : [int(train_matrix.shape[0]), int(train_matrix.shape[1])],
    'tfidf_sparsity_pct'  : round(100 * (1 - train_matrix.nnz / (train_matrix.shape[0] * train_matrix.shape[1])), 2),
    'idf_min'             : round(float(idf_values.min()), 4),
    'idf_max'             : round(float(idf_values.max()), 4),
    'idf_mean'            : round(float(idf_values.mean()), 4),
    'zipf_empirical_slope': round(float(slope), 3),
    'docs_empty_clean_text': int((df_proc['clean_tokens'] == 0).sum()),
}

with open(RESULTS_DIR / 'dataset_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

# Per-category split counts CSV
split_dist.reset_index().rename(columns={'index': 'category'}).to_csv(
    RESULTS_DIR / 'per_category_stats.csv', index=False
)

# Per-category top TF-IDF terms
tfidf_terms_rows = []
for cat, items in top_tfidf_per_cat.items():
    for rank, (term, score) in enumerate(items, 1):
        tfidf_terms_rows.append({'category': cat, 'rank': rank, 'term': term, 'mean_tfidf': score})
pd.DataFrame(tfidf_terms_rows).to_csv(RESULTS_DIR / 'top_tfidf_terms_per_category.csv', index=False)

print('Saved:')
for fname in ['dataset_stats.json', 'per_category_stats.csv', 'top_tfidf_terms_per_category.csv',
              'token_length_distribution.png', 'avg_tokens_per_category.png',
              'zipf_rank_frequency.png', 'top50_terms.png', 'top_terms_heatmap.png',
              'top10_terms_per_category.png', 'idf_distribution.png',
              'tfidf_weight_distribution.png', 'tfidf_top_terms_per_category.png',
              'tfidf_sparsity.png']:
    p = RESULTS_DIR / fname
    status = '✓' if p.exists() else '?'
    print(f'  [{status}] {fname}')

print()
print('Final summary:')
for k, v in stats.items():
    print(f'  {k:<30}: {v}')